# 00 · Calibrate: Assistant Axis reproduction on Kaggle

**Settings before running:** Accelerator = *GPU T4 x2*, Internet = *On*, and add the dataset that contains
`assistant-axis-code.zip` (our patched copy of `safety-research/assistant-axis`).
Run with **Save Version → Save & Run All** so outputs persist. Expected runtime: about 1–1.5 h.

What it does:
1. Reference numbers from the authors' released **Qwen3-32B** axis and role vectors (CPU only).
2. Tiny end-to-end run of the pipeline on **Qwen3-8B**: 6 roles × 5 prompts × 16 questions = 480 rollouts.
   generate (vLLM) → judge (Qwen3-14B-AWQ, served locally by vLLM) → activations (compact) → vectors → axis.
3. Times every stage and extrapolates the full run. Writes `/kaggle/working/calib_report.json`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
# Pinned versions (Kaggle image is Python 3.13; vLLM 0.10.2 ships abi3 wheels). vLLM brings its own torch.
import subprocess, sys
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'vllm==0.10.2', 'transformers==4.57.1', 'jsonlines', 'python-dotenv', 'openai==1.99.9'],
                   capture_output=True, text=True)
print(r.stdout[-1500:], r.stderr[-3000:])
assert r.returncode == 0, 'pip install failed'
print(subprocess.run([sys.executable, '-c', 'import vllm, transformers, torch, huggingface_hub as h; print("vllm", vllm.__version__, "transformers", transformers.__version__, "torch", torch.__version__, "hub", h.__version__)'],
                     capture_output=True, text=True).stdout)

Smoke-test vLLM on T4 (no bf16, compute 7.5). Falls back to the V0 engine if V1 can't run here.

In [ ]:
import os, subprocess, sys
SMOKE = ("from vllm import LLM, SamplingParams; "
         "llm = LLM('Qwen/Qwen3-0.6B', dtype='half', max_model_len=512, gpu_memory_utilization=0.5); "
         "print('OUT:', llm.generate(['Hello'], SamplingParams(max_tokens=8))[0].outputs[0].text)")
r = subprocess.run([sys.executable, '-c', SMOKE], capture_output=True, text=True)
if r.returncode != 0:
    print('V1 engine failed:', r.stderr[-2500:])
    os.environ['VLLM_USE_V1'] = '0'
    r = subprocess.run([sys.executable, '-c', SMOKE], capture_output=True, text=True, env=os.environ)
    print(r.stderr[-2500:])
assert r.returncode == 0, 'vLLM cannot run on this GPU'
print(r.stdout[-300:]); print('vLLM OK, VLLM_USE_V1 =', os.environ.get('VLLM_USE_V1', 'default'))

In [ ]:
import os
os.environ['HF_HUB_DISABLE_XET'] = '1'  # Xet token endpoint rate-limits (429) anonymous Kaggle IPs
os.environ.setdefault('HF_HUB_ETAG_TIMEOUT', '60')
import glob, os, shutil, subprocess, time, json, zipfile
hits = glob.glob('/kaggle/input/**/pipeline/1_generate.py', recursive=True)
if not hits:
    zips = glob.glob('/kaggle/input/**/assistant-axis-code*.zip', recursive=True)
    assert zips, 'Add the assistant-axis-code dataset to this notebook'
    zipfile.ZipFile(zips[0]).extractall('/kaggle/working/_src'); hits = glob.glob('/kaggle/working/_src/**/pipeline/1_generate.py', recursive=True)
SRC = os.path.dirname(os.path.dirname(hits[0]))
AA = '/kaggle/working/aa'
if not os.path.exists(AA): shutil.copytree(SRC, AA)
_g = f'{AA}/assistant_axis/generation.py'; _s = open(_g).read()
if 'VLLM_DTYPE' not in _s:
    _s = _s.replace("trust_remote_code=True,\n        )", "trust_remote_code=True,\n            dtype=os.environ.get('VLLM_DTYPE', 'auto'),\n        )", 1)
    if 'import os' not in _s: _s = 'import os\n' + _s
    open(_g, 'w').write(_s)
assert 'VLLM_DTYPE' in open(_g).read(), 'dtype patch did not apply'
os.environ['VLLM_DTYPE'] = 'half'
_m = f'{AA}/assistant_axis/internals/model.py'; _s = open(_m).read()
_anchor = "self.model = AutoModelForCausalLM.from_pretrained(model_name, **model_kwargs)\n"
if 'NO_OFFLOAD' not in _s:
    assert _anchor in _s, 'model.py anchor not found'
    _s = _s.replace(_anchor, _anchor + "        # NO_OFFLOAD: fail fast instead of silently running layers on CPU/disk\n"
        "        _bad = {k: v for k, v in (getattr(self.model, 'hf_device_map', None) or {}).items() if str(v) in ('cpu', 'disk')}\n"
        "        if _bad: raise RuntimeError(f'model offloaded to CPU/disk (GPUs not free?): {list(_bad)[:5]}')\n", 1)
    open(_m, 'w').write(_s)
assert 'NO_OFFLOAD' in open(_m).read(), 'offload guard did not apply'
# transformers 5.x returns BatchEncoding from apply_chat_template(tokenize=True); force plain token lists
for _f in glob.glob(f'{AA}/assistant_axis/**/*.py', recursive=True):
    _s = open(_f).read()
    if 'tokenize=True,' in _s and 'return_dict=False' not in _s:
        open(_f, 'w').write(_s.replace('tokenize=True,', 'tokenize=True, return_dict=False,'))
def gpu_used_mib():
    q = subprocess.run(['nvidia-smi', '--query-gpu=memory.used', '--format=csv,noheader,nounits'], capture_output=True, text=True)
    return [int(x) for x in q.stdout.split()]
def wait_gpus_free(limit_mib=1500, timeout=300):
    for _ in range(timeout // 5):
        u = gpu_used_mib()
        if max(u) < limit_mib: print('GPUs free:', u, 'MiB used'); return
        time.sleep(5)
    subprocess.run('nvidia-smi', shell=True)
    raise RuntimeError(f'GPUs still busy: {gpu_used_mib()} MiB used')
PIPE = f'{AA}/pipeline'
OUT = '/kaggle/working/calib'; os.makedirs(OUT, exist_ok=True)
REPORT = {}
TARGET = 'Qwen/Qwen3-8B'; JUDGE = 'Qwen/Qwen3-14B-AWQ'; TARGET_LAYER = 18
ROLES = ['default', 'assistant', 'pirate', 'demon', 'oracle', 'accountant']
QCOUNT = 16
def run(cmd, env=None):
    t = time.time(); print('$', cmd)
    r = subprocess.run(cmd, shell=True, cwd=PIPE, env={**os.environ, **(env or {})})
    assert r.returncode == 0, f'failed: {cmd}'
    return time.time() - t
print(SRC)

## 1. Reference: released Qwen3-32B axis and role vectors

In [ ]:
import torch, numpy as np
from huggingface_hub import snapshot_download
ref = None
for attempt in range(6):
    try:
        ref = snapshot_download('lu-christina/assistant-axis-vectors', repo_type='dataset',
                                allow_patterns=['qwen-3-32b/*'], max_workers=2)
        break
    except Exception as e:
        print(f'download attempt {attempt + 1} failed: {str(e)[:200]}'); time.sleep(30 * (attempt + 1))
if ref is None:
    print('WARNING: reference vectors unavailable (HF rate limit); skipping section 1')
    files = []
else:
    files = sorted(glob.glob(f'{ref}/qwen-3-32b/**/*', recursive=True)); print(len(files), 'files'); print(*[f.replace(ref,'') for f in files[:15]], sep='\n')

In [ ]:
if ref is not None:
    def load_any(p):
        x = torch.load(p, map_location='cpu', weights_only=False)
        return x['vector'] if isinstance(x, dict) and 'vector' in x else x
    axis32 = load_any(f'{ref}/qwen-3-32b/assistant_axis.pt').float()       # (64, 5120)
    L32 = 32
    vec_files = [f for f in files if f.endswith('.pt') and '/assistant_axis' not in f and 'capping' not in f]
    names, vecs = [], []
    for f in vec_files:
        v = load_any(f)
        if torch.is_tensor(v) and v.dim() == 2 and v.shape == axis32.shape:
            names.append(os.path.relpath(f, f'{ref}/qwen-3-32b')); vecs.append(v.float()[L32])
    is_role = np.array(['role' in n for n in names]); print('roles:', is_role.sum(), 'traits:', (~is_role).sum())
    names = [n for n, r in zip(names, is_role) if r]; vecs = [v for v, r in zip(vecs, is_role) if r]
    V = torch.stack(vecs); print('role vectors at layer', L32, V.shape)
    Vc = V - V.mean(0)
    U, S, Vt = torch.linalg.svd(Vc, full_matrices=False)
    ev = (S**2) / (S**2).sum()
    pc1 = Vt[0]; a = axis32[L32] / axis32[L32].norm()
    cos = float(torch.dot(pc1, a)); pc1 = pc1 * np.sign(cos)
    proj = (V @ a).numpy(); order = np.argsort(proj)
    REPORT['ref_qwen32b'] = {
      'n_vectors': len(names), 'layer': L32,
      'cos_pc1_axis': abs(cos), 'var_pc1': float(ev[0]), 'n_pcs_70pct': int((ev.cumsum(0) < 0.7).sum() + 1),
      'axis_norm_by_layer': axis32.norm(dim=1).tolist(),
      'most_assistant_like': [names[i] for i in order[::-1][:10]], 'least_assistant_like': [names[i] for i in order[:10]],
    }
    print(json.dumps({k: v for k, v in REPORT['ref_qwen32b'].items() if k != 'axis_norm_by_layer'}, indent=1))

## 2. Generate (Qwen3-8B, vLLM, TP=2)

In [ ]:
t = run(f"python 1_generate.py --model {TARGET} --output_dir {OUT}/responses --question_count {QCOUNT} "
        f"--tensor_parallel_size 2 --gpu_memory_utilization 0.90 --roles {' '.join(ROLES)}")
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(TARGET)
import jsonlines
n, ntok, hit_cap = 0, 0, 0
samples = []
for f in sorted(glob.glob(f'{OUT}/responses/*.jsonl')):
    for e in jsonlines.open(f):
        r = [m['content'] for m in e['conversation'] if m['role'] == 'assistant'][0]
        k = len(tok(r).input_ids); n += 1; ntok += k; hit_cap += k >= 510
        samples.append((os.path.basename(f)[:-6], f"{e['label']}_p{e['prompt_index']}_q{e['question_index']}", e['question'], r))
REPORT['gen'] = {'seconds': t, 'rollouts': n, 'out_tokens': ntok, 'mean_tokens': ntok / n,
                 'frac_hit_512_cap': hit_cap / n, 'rollouts_per_hour': n / t * 3600}
print(REPORT['gen'])

## 3. Judge (Qwen3-14B-AWQ served by vLLM)

In [ ]:
srv = subprocess.Popen(f"python -m vllm.entrypoints.openai.api_server --model {JUDGE} --tensor-parallel-size 2 "
                       f"--max-model-len 4096 --dtype half --gpu-memory-utilization 0.90 --port 8000",
                       shell=True, stdout=open('/kaggle/working/judge_server.log', 'w'), stderr=subprocess.STDOUT,
                       start_new_session=True)  # own process group so we can kill vLLM + its TP workers
import urllib.request
for _ in range(180):
    try: urllib.request.urlopen('http://localhost:8000/health'); break
    except Exception: time.sleep(10)
else: raise RuntimeError(open('/kaggle/working/judge_server.log').read()[-3000:])
JENV = {'OPENAI_BASE_URL': 'http://localhost:8000/v1', 'OPENAI_API_KEY': 'EMPTY', 'JUDGE_TEMPERATURE': '0'}
t = run(f"python 3_judge.py --responses_dir {OUT}/responses --output_dir {OUT}/scores --judge_model {JUDGE} "
        f"--batch_size 64 --requests_per_second 200", JENV)
t2 = run(f"python 3_judge.py --responses_dir {OUT}/responses --output_dir {OUT}/scores --judge_model {JUDGE}", JENV)  # retry unparsed
import signal
def stop_judge():
    for sig in (signal.SIGTERM, signal.SIGKILL):
        try: os.killpg(srv.pid, sig)
        except ProcessLookupError: break
        try: srv.wait(timeout=60); break
        except subprocess.TimeoutExpired: pass
    subprocess.run("pkill -9 -f 'vllm.entrypoints' ; pkill -9 -f 'VLLM::' ", shell=True)
    wait_gpus_free()
stop_judge()
from collections import Counter
dist, judged = {}, 0
for f in sorted(glob.glob(f'{OUT}/scores/*.json')):
    s = json.load(open(f)); judged += len(s); dist[os.path.basename(f)[:-5]] = dict(Counter(s.values()))
REPORT['judge'] = {'seconds': t + t2, 'judged': judged, 'score_dist': dist}
print(json.dumps(REPORT['judge'], indent=1))

## 4. Activations (fp16 on T4, compact storage) → vectors → axis

In [ ]:
wait_gpus_free()
t = run(f"python 2_activations.py --model {TARGET} --responses_dir {OUT}/responses --output_dir {OUT}/activations "
        f"--dtype float16 --save_dtype float16 --scores_dir {OUT}/scores --keep_layers {TARGET_LAYER} "
        f"--batch_size 2 --tensor_parallel_size 2",
        {'CUDA_LAUNCH_BLOCKING': '1'})  # batch 8 on the 2xT4 split model hits an async "unspecified launch failure"; both fixes verified in aa-debug-act
sizes = {os.path.basename(f): os.path.getsize(f) for f in glob.glob(f'{OUT}/activations/*.pt')}
REPORT['act'] = {'seconds': t, 'bytes_per_role': sum(sizes.values()) / len(sizes)}
d = torch.load(f'{OUT}/activations/pirate.pt', weights_only=False)
print('pirate: n_all', d['n_all'], 'n_score3', d['n_score3'], 'mean_all finite:', bool(torch.isfinite(d['mean_all']).all()))
run(f"python 4_vectors.py --activations_dir {OUT}/activations --scores_dir {OUT}/scores --output_dir {OUT}/vectors --min_count 5")
run(f"python 5_axis.py --vectors_dir {OUT}/vectors --output {OUT}/axis.pt")
ax8 = torch.load(f'{OUT}/axis.pt', weights_only=False).float()
REPORT['act']['axis_norm_by_layer'] = ax8.norm(dim=1).tolist()
print(REPORT['act'])

## 5. Extrapolate the full run and save the report

In [ ]:
N_ROLES = 276
est = {}
for q in (24, 40, 48):
    n = N_ROLES * 5 * q
    est[f'q{q}'] = {'rollouts': n,
        'gen_h': n / REPORT['gen']['rollouts'] * REPORT['gen']['seconds'] / 3600,
        'judge_h': n / max(REPORT['judge']['judged'], 1) * REPORT['judge']['seconds'] / 3600,
        'act_h': n / REPORT['gen']['rollouts'] * REPORT['act']['seconds'] / 3600,
        'act_GB': N_ROLES * REPORT['act']['bytes_per_role'] * q / QCOUNT / 1e9}
    est[f'q{q}']['total_h'] = est[f'q{q}']['gen_h'] + est[f'q{q}']['judge_h'] + est[f'q{q}']['act_h']
REPORT['estimate_full'] = est
import random; random.seed(0)
def score_of(role, key):
    f = f'{OUT}/scores/{role}.json'
    return json.load(open(f)).get(key) if os.path.exists(f) else None
REPORT['spot_check'] = [{'role': r, 'key': k, 'q': q, 'response': a[:1200], 'judge_score': score_of(r, k)}
                        for r, k, q, a in random.sample(samples, 40)]
json.dump(REPORT, open('/kaggle/working/calib_report.json', 'w'), indent=1)
# keep outputs small: drop the scratch copies
shutil.rmtree('/kaggle/working/_src', ignore_errors=True)
print(json.dumps(est, indent=1))